In [2]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

con = sqlite3.connect(r'C:\Users\faraz\OneDrive\Documents\SEM 3  AIRTEL DATA\Airtel_Telecom_Analysis\data\raw\airtel.db')
def q(sql):
    return pd.read_sql(sql, con)

In [3]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [4]:
def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
    pool = (k1 + k2) / (n1 + n2)               # both groups as one
    se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
    z = (k1/n1 - k2/n2) / se                   # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [5]:
# Claim :the two groups used the same amount of data before the offer 
# H₀ :both groups have the same true average daily data before May
# H₁ :the two averages differ
# Tail :two-tailed — a gap either way would break the split

In [6]:
pre = q("""
SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;
""")

In [7]:
a=pre[pre.g=='A'].m
b=pre[pre.g=='B'].m


f_test(a, b)  # check if the two groups have the same variance

F = 1.1108  p = 3.673e-08


In [8]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)

stated_arpu
tenure_days
complaint_count


In [9]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    f_test(B[col], A[col])
    

stated_arpu
F = 0.9374  p = 0.0006932
tenure_days
F = 0.9762  p = 0.2061
complaint_count
F = 0.9765  p = 0.2128


In [10]:
rev = q("""
SELECT
    c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r
    ON r.customer_id = c.customer_id
    AND r.recharge_date >= '2025-01-01'
    AND r.recharge_date < '2025-05-01'
GROUP BY c.customer_id;
""")
rev

,g,s
0,B,897
1,A,198
2,A,3408
3,B,0
4,B,1438
...,...,...
21995,A,6891
21996,A,719
21997,B,836
21998,A,798


In [11]:
A = rev[rev.g == 'A'].s
B = rev[rev.g == 'B'].s

f_test(B, A)

#p-value < 0.05 Hence significent
#WE can do Z_test


z_mean2(B,A)

#p_value>0.05 hence not significent
#FAILD TO REJECTh0

F = 0.9513  p = 0.008866
z = -1.3490  p = 0.1773


In [12]:
q("""

SELECT ab_group, 
       COUNT(*) AS n,
       SUM(device_type = 'Smartphone') AS smart,
       SUM(connection_type = 'Prepaid') AS prepaid
FROM customer_features 
GROUP BY ab_group;

""")

,ab_group,n,smart,prepaid
0,A,10996,9417,8575
1,B,11004,9554,8574


In [13]:
 # Smartphone 
z_prop2(9554, 11004, 9417, 10996)

#z_prop2(succes_of_B,total_b,succes_of_A,total_a)

#Prepaid 
z_prop2(8574, 11004, 8575, 10996)

z = 2.5457  p = 0.01091
z = -0.1177  p = 0.9063


In [14]:
# Claim : the offer raised daily data usage
# H₀ :both groups have the same true average daily data in May and June
# H₁ :group B's average is higher
# Tail :one-tailed — the company said so in advance


off= q("""

SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM   usage_daily u
JOIN   customer_features c ON c.customer_id = u.customer_id
WHERE  u.usage_date >= '2025-05-01'
GROUP  BY u.customer_id;

""")

In [15]:
A=off[off.g=='A'].m
B=off[off.g=='B'].m

In [16]:
q("""
SELECT ab_group, COUNT(*) AS n,
       SUM(is_churned) AS all_churn,
       SUM(churn_date >= '2025-05-01') AS may_on
FROM   customer_features GROUP BY ab_group;
""")

,ab_group,n,all_churn,may_on
0,A,10996,2587,905
1,B,11004,2639,914


In [17]:
# Claim :the offer raised revenue
# H₀ :both groups paid the same on average
# H₁ :group B paid more
# Tail : one-tailed — the company said so in advance
Cev = q("""
SELECT c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r ON r.customer_id=c.customer_id
    AND r.recharge_date >= '2025-05-01'
GROUP BY c.customer_id;
""")

Cev

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


In [18]:
a=Cev[Cev.g=='A'].s
b=Cev[Cev.g=='B'].s
a,b

(1           0
 2        1665
 5         396
 6        3576
 13       3529
          ... 
 21989       0
 21995    3587
 21996       0
 21998       0
 21999       0
 Name: s, Length: 10996, dtype: int64,
 0          0
 3          0
 4          0
 7          0
 8        349
         ... 
 21991    992
 21992    349
 21993      0
 21994      0
 21997    209
 Name: s, Length: 11004, dtype: int64)

In [19]:
f_test(b,a)

F = 0.9697  p = 0.1068


In [20]:
stats.ttest_ind(b,a,alternative="greater")

# p value.float64(0.7232)

# p-value > 0.05 hence not significant
# Hence failed to reject H₀

TtestResult(statistic=np.float64(-0.5926646653701372), pvalue=np.float64(0.7232941573761509), df=np.float64(21998.0))

In [21]:
q("""
 select connection_type as grp,
  count(*) as n, sum(is_churned) as k
  from customer_features
  group by connection_type;
  """)

,grp,n,k
0,Postpaid,4851,785
1,Prepaid,17149,4441


In [22]:
q("""select connection_type as grp, count(*) as n, sum(is_churned) as k
from customer_features
group by connection_type;""")

z_prop2(4441, 17149, 785, 4851)
# p-value < 0.05 so its significant
# HENCE REJECT h0

z = 14.0364  p = 9.333e-45


In [23]:
q("""
select case when complaint_count=0 then 'neever'
when unresolved_count =0 then 'all closed'
else 'open' end as grp,
count(*) as n, sum(is_churned) as k
from customer_features group by grp;
""")
z_prop2(1834, 6680, 1328, 6631)
# p-value <  so result is significant
# HENCE REJECT h0

z = 10.0684  p = 7.623e-24


In [24]:
# Claim :year-0 and year-3 subscribers churn at different rates
# H₀ :they share one true churn rate
# H₁ :they do not
q("""
SELECT tenure_years AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY tenure_years;
""")
z_prop2(1328,4610,632,3374)
# p-value<0.05 so its significant
# hence reject ho

z = 10.3330  p = 4.995e-25


In [25]:
# Claim :Himachal Pradesh and Punjab churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT circle_name AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY circle_name;
""")
z_prop2(417,1247,77,558)

z = 8.6494  p = 5.176e-18


In [26]:
# Claim :Himachal Pradesh and Punjab churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT circle_name AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY circle_name;
""")
z_prop2(417,1247,77,558)

z = 8.6494  p = 5.176e-18


In [27]:
q("""select device_type as grp,
count(*) as n, sum(is_churned) as k
from customer_features
group by device_type;""")

z_prop2(4507, 18971, 719, 3029)

z = 0.0241  p = 0.9807


In [28]:
q("""select c.gender, c.device_type,
u.data_mb as m
from usage_daily u
join customer_features c on c.customer_id = u.customer_id;""")

,gender,device_type,m
0,Male,Smartphone,213.7
1,Male,Smartphone,1541.0
2,Male,Smartphone,107.4
3,Male,Smartphone,1569.6
4,Male,Smartphone,845.9
...,...,...,...
1326734,Male,Smartphone,335.9
1326735,Male,Smartphone,2465.5
1326736,Male,Smartphone,405.0
1326737,Male,Smartphone,1192.0


In [29]:
# Claim :gender and device type each change the data a subscriber uses
# H₀ :both groups have the same true average daily data
# H₁ :the two averages differ

gen = q("""
SELECT c.gender, c.device_type,
u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id;
""")
dev = q("""
SELECT c.gender, c.device_type,
AVG(u.data_mb) AS m
from usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY c.customer_id;
""")

In [30]:
male = gen[gen.gender=='Male'].m
female = gen[gen.gender=='Female'].m
female

f_test(female, male)
z_mean2(female, male)

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [31]:
fp = dev[dev.device_type=='Feature Phone'].m
sp = dev[dev.device_type=='Smartphone'].m
f_test(fp, sp)
z_mean2(fp, sp)

F = 0.0268  p = 0
z = -119.2400  p = 0


In [34]:
# Claim : subscribers with complaints churn at different rates
# H₀ : both groups have the same true churn rate
# H₁ : the two rates differ

q("""
SELECT 
CASE WHEN complaint_count = 0 THEN 'No complaints'
ELSE 'Complaints' END AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY grp;
""")

z_prop2(1834,6680,1328,6631)

# p-value > 0.05 so significant
# hence reject H₀

z = 10.0684  p = 7.623e-24


In [37]:
# Claim: Subscribers with high data usage churn at different rates compared to subscribers with low data usage.
# H₀: Both groups have the same true churn rate.
# H₁: The two churn rates differ.
q("""
SELECT 
CASE WHEN roaming_days = 0 THEN 'No Roaming'
ELSE 'Roaming' END AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY grp;
""")
z_prop2(2653,16233,2573,5767)
# p-value < 0.05 so significant
# hence reject H₀

z = -43.3361  p = 0
